# Exploração da API do Comex Stat (MDIC)

Objetivo: identificar como obter os valores mensais de:
- Exportações (US$ bi)
- Importações (US$ bi)
- Saldo Comercial (US$ bi)

Fontes candidatas:
- API oficial: https://api-comexstat.mdic.gov.br/
- Alternativa: BCB (SGS) — séries 22707, 22708, 22709

In [4]:
import requests

# Testar a API do Comex Stat — endpoint de metadados
URL_COMEX = "https://api-comexstat.mdic.gov.br/general/details"

# Primeiro, uma chamada OPTIONS para ver se o endpoint existe
r = requests.options(URL_COMEX, timeout=10)
print(f"OPTIONS — Status: {r.status_code}")
print(f"Headers: {dict(r.headers)}")
print()

# Uma chamada GET simples
r = requests.get(URL_COMEX, timeout=10)
print(f"GET — Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()
print(r.text[:1000])

OPTIONS — Status: 403
Headers: {'Date': 'Sun, 20 Sep 2026 15:52:13 GMT', 'Content-Type': 'text/html; charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Cache-Control': 'private, max-age=0, no-store, no-cache, must-revalidate, post-check=0, pre-check=0', 'Expires': 'Thu, 01 Jan 1970 00:00:01 GMT', 'Referrer-Policy': 'same-origin', 'X-Frame-Options': 'SAMEORIGIN', 'Content-Encoding': 'gzip', 'Server': 'cloudflare', 'CF-RAY': 'a3e1fefc58e3a356-GIG'}

GET — Status: 200
Content-Type: application/json

{"data":{"list":[{"filter":"country","text":"Pa\u00edses"},{"filter":"economicBlock","text":"Blocos"},{"filter":"state","text":"UF do produto"},{"filter":"via","text":"Via"},{"filter":"urf","text":"URF"},{"filter":"ncm","text":"NCM - Nomenclatura Comum do Mercosul"},{"filter":"subHeading","text":"Subposi\u00e7\u00e3o (SH6)"},{"filter":"heading","text":"Posi\u00e7\u00e3o (SH4)"},{"filter":"chapter","text":"Cap\u00edtulo (SH2)"},{"filter":"section","text":"Se\u00e7\u00e

In [ ]:
import requests

URL_COMEX = "https://api-comexstat.mdic.gov.br"

r = requests.get(
    URL_COMEX,
    timeout=10
)

print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print(r.text[:2000])

In [5]:
# Testar POST /general para exportações
URL_GENERAL = "https://api-comexstat.mdic.gov.br/general"

body = {
    "flow": "export",
    "monthDetail": True,
    "period": {"from": "2026-06", "to": "2026-07"},
    "filters": [],
    "details": [],
    "metrics": ["metricFOB"],
}

r = requests.post(URL_GENERAL, json=body, timeout=20)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()

if r.status_code == 200:
    dados = r.json()
    print(f"Chaves do retorno: {list(dados.keys())}")
    print()
    print("Conteúdo (primeiras 2000 chars):")
    print(json.dumps(dados, indent=2, ensure_ascii=False)[:2000])
else:
    print(f"Erro: {r.text[:500]}")

Status: 200
Content-Type: application/json

Chaves do retorno: ['data', 'success', 'message', 'processo_info', 'language']

Conteúdo (primeiras 2000 chars):


NameError: name 'json' is not defined

In [6]:
# Ver o retorno sem formatar
print("Tipo:", type(r.json()))
print()
print("Chaves e tipos dos valores:")
for chave, valor in r.json().items():
    print(f"  {chave!r}: {type(valor).__name__} → {repr(valor)[:200]}")

Tipo: <class 'dict'>

Chaves e tipos dos valores:
  'data': dict → {'list': [{'year': '2026', 'monthNumber': '06', 'metricFOB': '35492956607'}, {'year': '2026', 'monthNumber': '07', 'metricFOB': '33723397019'}]}
  'success': bool → True
  'message': NoneType → None
  'processo_info': NoneType → None
  'language': str → 'pt'


In [7]:
# Testar com período mais antigo (2024)
body = {
    "flow": "export",
    "monthDetail": True,
    "period": {"from": "2024-01", "to": "2024-03"},
    "filters": [],
    "details": [],
    "metrics": ["metricFOB"],
}

r = requests.post(URL_GENERAL, json=body, timeout=20)
print(f"Status: {r.status_code}")
dados = r.json()

print(f"\nChave 'data': {type(dados.get('data'))}")
print(f"Conteúdo de 'data': {repr(dados.get('data'))[:1500]}")

Status: 200

Chave 'data': <class 'dict'>
Conteúdo de 'data': {'list': [{'year': '2024', 'monthNumber': '03', 'metricFOB': '27657419417'}, {'year': '2024', 'monthNumber': '01', 'metricFOB': '26702655353'}, {'year': '2024', 'monthNumber': '02', 'metricFOB': '23348254025'}]}


In [8]:
# Testar com período bem amplo
body = {
    "flow": "export",
    "monthDetail": False,  # False = agrega por ano
    "period": {"from": "2024-01", "to": "2024-12"},
    "filters": [],
    "details": [],
    "metrics": ["metricFOB"],
}

r = requests.post(URL_GENERAL, json=body, timeout=20)
dados = r.json()
print(f"Status: {r.status_code}")
print(f"data: {repr(dados.get('data'))[:1500]}")

Status: 200
data: {'list': [{'year': '2024', 'metricFOB': '337046161710'}]}


In [9]:
dados = r.json()
print("Mensagem:", dados.get("message"))
print("Sucesso:", dados.get("success"))
print("Processo:", dados.get("processo_info"))

Mensagem: None
Sucesso: True
Processo: None
